# 03 · FunnyBirds + MCBM — does minimality fix grounding?  *(seed-aware)*

**Claim (MCBM):** IB makes `z_j` a minimal sufficient statistic of `c_j`. Loss
`L = CE + λ_c·BCE(z,c) + γ·0.2·mean((6c−3 − z)²)`. **Hypothesis:** minimality constrains
*content*, backwash is about *source*; when `c=f(class)` the ±3 target is class-derived,
so tightening γ can't remove class-reading. Decision rule + null criteria: `DECISIONS §D.5`.
All cells aggregate over seeds. *Refs: `fb_mcbm_renderer_swap.ipynb`, `fb_mcbm_rl_renderer_swap.ipynb`.*

In [ ]:
import os, json, re, glob
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
CURATED = Path(os.environ["CURATED_DATA"]); REPO = Path.cwd().parent
import sys; sys.path.insert(0, str(REPO/"analysis"))
try:
    from plotting import set_paper_style, PALETTE; set_paper_style()
    CBM_C, MCBM_C = PALETTE["CBM"], PALETTE["MCBM"]
except Exception:
    CBM_C, MCBM_C = "#0072B2", "#D55E00"
plt.rcParams["figure.dpi"]=120
EPS = 1e-3
def parse_stem(stem):
    m=re.match(r"^funnybirds-(vanilla|cbm|mcbm)(?:-g([0-9p]+))?-s(\d+)$", stem)
    if not m: return None
    gamma=float(m.group(2).replace("p",".")) if m.group(2) else np.nan
    return m.group(1), gamma, int(m.group(3))
def need(p, how):
    ok=Path(p).exists()
    if not ok: print(f"[pending] {p}\n  produce it:  {how}")
    return ok
# ---- seed-aware grounding loaders ----
def load_grounding(prefix):
    """All seeds for a config prefix -> one df with a 'seed' column (None if absent)."""
    fs = sorted(glob.glob(str(CURATED/"grounding"/f"{prefix}-s*.parquet")))
    if not fs: return None
    out=[]
    for f in fs:
        d=pd.read_parquet(f); d["seed"]=int(re.search(r"-s(\d+)\.parquet$", f).group(1)); out.append(d)
    return pd.concat(out, ignore_index=True)
def per_part_seedagg(df, visible_only=True):
    """per (seed,part) retained_frac -> per-part mean/std/count across seeds."""
    d = df[df["changed_frac"]>EPS] if (visible_only and "changed_frac" in df.columns) else df
    g = d.groupby(["seed","part"]).agg(pi=("p_intact","mean"), pr=("p_removed","mean"))
    g["rf"] = g.pr/g.pi
    return g["rf"].groupby("part").agg(["mean","std","count"])


## 1 · Overall `retained_frac` vs γ (±seed std) — the summary (diluted)
`backwash_vs_gamma.csv` (collected on **visible-only** rows). Averages all 5 parts, so
sits near 0.1 regardless — §2 is the one to show. Error bars = std across seeds.

In [ ]:
bw = CURATED/"backwash_vs_gamma.csv"
if need(bw, 'bash analysis/grounding_sweep.sh'):
    T = pd.read_csv(bw); display(T.round(3))
    mc = T[(T.model=="mcbm") & T.retained_frac.notna()]; cb = T[T.model=="cbm"]
    g = mc.groupby("gamma").retained_frac.agg(["mean","std"]).reset_index()
    floor=(g.gamma[g.gamma>0].min() or 0.05)/3
    fig,ax=plt.subplots(figsize=(6.2,4))
    ax.errorbar(g.gamma.replace(0,floor), g["mean"], yerr=g["std"].fillna(0), marker="o", capsize=3, color=MCBM_C, label="MCBM")
    if len(cb): ax.axhline(cb.retained_frac.mean(), ls="--", color=CBM_C, label="CBM (ref)")
    ax.set_xscale("log"); ax.set_xlabel("γ (effective force = γ×0.2)"); ax.set_ylabel("overall retained_frac")
    ax.set_ylim(0,1.02); ax.set_title("Overall removed-part retention vs γ"); ax.legend()
    print("seeds per gamma:", mc.groupby("gamma").seed.nunique().to_dict())

📊 Overall retained_frac is **flat across γ (~0.09–0.11) and ≈ CBM** — but it is diluted by the 4 near-zero parts; §2 is the real view.

## 2 · Per-part `retained_frac` vs γ — **the figure**  (tail, ±seed band)
Read every seed's grounding parquet, visible-only, break out by part. Does the **tail**
curve come down as γ rises? (Shaded = ±std across seeds.)

In [ ]:
recs=[]
for f in sorted(glob.glob(str(CURATED/"grounding"/"funnybirds-*-s*.parquet"))):
    pr=parse_stem(Path(f).stem)
    if pr is None or pr[0]=="vanilla": continue
    model,gamma,seed=pr; d=pd.read_parquet(f)
    if "changed_frac" in d.columns: d=d[d["changed_frac"]>EPS]
    gg=d.groupby("part").agg(pi=("p_intact","mean"),pr=("p_removed","mean"))
    for part,row in gg.iterrows(): recs.append(dict(model=model,gamma=gamma,seed=seed,part=part,retained_frac=row.pr/row.pi))
P=pd.DataFrame(recs)
if len(P):
    parts=["tail","wing","beak","foot","eye"]
    A=P.groupby(["model","gamma","part"]).retained_frac.agg(["mean","std"]).reset_index()
    mc=A[A.model=="mcbm"]; cbm_pp=A[A.model=="cbm"].set_index("part")["mean"]
    floor=(mc.gamma[mc.gamma>0].min() or 0.05)/3
    fig,ax=plt.subplots(figsize=(7,4.4)); cmap=plt.cm.viridis(np.linspace(0,0.85,len(parts)))
    for part,col in zip(parts,cmap):
        s=mc[mc.part==part].sort_values("gamma");
        if not len(s): continue
        x=s.gamma.replace(0,floor)
        ax.plot(x,s["mean"],"o-",color=col,lw=3 if part=="tail" else 1.4,label=part+("  ← highest retention" if part=="tail" else ""))
        ax.fill_between(x, s["mean"]-s["std"].fillna(0), s["mean"]+s["std"].fillna(0), color=col, alpha=0.15)
        if part in cbm_pp.index: ax.scatter([floor/1.6],[cbm_pp[part]],marker="*",s=90,color=col,zorder=5)
    ax.set_xscale("log"); ax.set_xlabel("γ (effective force = γ×0.2)"); ax.set_ylabel("retained_frac (visible-only)")
    ax.set_ylim(-0.02,1.02); ax.set_title("Per-part retention vs γ (★=CBM)\nminimality does not bring tail down"); ax.legend(title="part",fontsize=8)
    print("tail retained_frac by γ (mean±std over seeds):")
    display(mc[mc.part=="tail"][["gamma","mean","std"]].sort_values("gamma").round(3))
else: print("[pending] bash analysis/grounding_sweep.sh")

## 3 · Species-code vs γ (seed-averaged) — did the class channel survive?
If `species←c_preds` stays ≈1 as γ grows, minimality compressed the representation
without cutting the class channel.

In [ ]:
rows=[]
for f in sorted(glob.glob(str(CURATED/"species_probe"/"funnybirds-mcbm-g*-s*.json"))):
    m=re.search(r"-g([0-9p]+)-s(\d+)\.json$", Path(f).name)
    if not m: continue
    S=json.loads(Path(f).read_text())
    rows.append(dict(gamma=float(m.group(1).replace("p",".")), seed=int(m.group(2)),
                     c=S["species_from_cpreds"]["acc"], tail=S["species_from_part_cpreds"].get("tail",{}).get("acc",np.nan),
                     chance=S["chance"]))
if rows:
    D=pd.DataFrame(rows); Dg=D.groupby("gamma").agg(c=("c","mean"),tail=("tail","mean"),chance=("chance","first")).reset_index()
    display(Dg.round(3)); floor=(Dg.gamma[Dg.gamma>0].min() or 0.05)/3
    cbj=sorted(glob.glob(str(CURATED/"species_probe"/"funnybirds-cbm-s*.json")))
    fig,ax=plt.subplots(figsize=(6.2,4))
    ax.plot(Dg.gamma.replace(0,floor),Dg.c,"o-",color=MCBM_C,label="species←c_preds")
    ax.plot(Dg.gamma.replace(0,floor),Dg["tail"],"s--",color="#5B8C5A",label="species←tail concepts")
    if cbj: ax.axhline(np.mean([json.loads(Path(p).read_text())["species_from_cpreds"]["acc"] for p in cbj]),ls=":",color=CBM_C,label="CBM species←c_preds")
    ax.axhline(Dg.chance.iloc[0],ls=":",color="k",label="chance"); ax.set_xscale("log")
    ax.set_xlabel("γ"); ax.set_ylabel("species recoverable"); ax.set_ylim(0,1.02); ax.legend(fontsize=8)
    ax.set_title("Class channel survives minimality")
else: print("[pending] grounding_sweep.sh runs the probe too")

📊 **species←c_preds stays ~0.99 across γ** → tightening minimality compresses the code but leaves the class channel intact.

## 4 · CONTROL — did γ actually tighten the bottleneck? (seed-averaged)
Flat retention only refutes minimality **if γ changed the representation**. Read `z` from
saved predictions: `mean((6c−3 − z)²)` ↓ / `mean|z|` ↑ with γ = γ bit. See `DECISIONS §D.5`.

In [ ]:
import torch
def zstats_seed(cfg, seed):
    pth = REPO/"external"/"minimal_cbm"/"results"/cfg/str(seed)/"predictions"/"epoch_100.pth"
    if not pth.exists(): return None
    d=torch.load(pth, map_location="cpu", weights_only=False); z,cc=d["z"].float(),d["c"].float()
    if not (np.isfinite(z).all() and np.isfinite(cc).all()): return None
    yp=d["y_preds"]; ta=float((yp.argmax(-1)==d["y"]).float().mean()) if yp.ndim>1 else float((yp==d["y"]).float().mean())
    cp=d["c_preds"]; cp=cp[...,0] if cp.ndim==3 else cp
    return float(((6*cc-3-z)**2).mean()), float(z.abs().mean()), ta, float(((cp>=0.5).float()==cc).float().mean())
rows=[]
for g,tag in [(0,"g0"),(0.1,"g0p1"),(0.3,"g0p3"),(1,"g1"),(3,"g3"),(5,"g5")]:
    for seed in range(1,6):
        s=zstats_seed(f"funnybirds-mcbm-{tag}", seed)
        if s: rows.append((g,seed,*s))
if rows:
    D=pd.DataFrame(rows,columns=["gamma","seed","rep_loss","mean_abs_z","task_acc","concept_acc"])
    Dg=D.groupby("gamma").agg(rep_loss=("rep_loss","mean"),mean_abs_z=("mean_abs_z","mean"),
                              task_acc=("task_acc","mean"),concept_acc=("concept_acc","mean"),n_seeds=("seed","nunique")).reset_index()
    display(Dg.round(3)); floor=(Dg.gamma[Dg.gamma>0].min() or 0.05)/3
    fig,ax=plt.subplots(1,2,figsize=(11,3.8))
    ax[0].plot(Dg.gamma.replace(0,floor),Dg.rep_loss,"o-",color=MCBM_C,label="mean (±3−z)²")
    a0=ax[0].twinx(); a0.plot(Dg.gamma.replace(0,floor),Dg.mean_abs_z,"s--",color="#5B8C5A",label="mean|z|")
    ax[0].set_xscale("log"); ax[0].set_xlabel("γ"); ax[0].set_ylabel("minimality term (↓=pinned)"); a0.set_ylabel("mean|z|"); ax[0].set_title("Did γ bite?")
    ax[1].plot(Dg.gamma.replace(0,floor),Dg.task_acc,"o-",label="task"); ax[1].plot(Dg.gamma.replace(0,floor),Dg.concept_acc,"s--",label="concept")
    ax[1].set_xscale("log"); ax[1].set_xlabel("γ"); ax[1].set_ylabel("val acc"); ax[1].set_ylim(0,1.02); ax[1].legend(); ax[1].set_title("Fit vs γ")
    plt.tight_layout()
    moved=(Dg.rep_loss.max()-Dg.rep_loss.min())>0.05*max(Dg.rep_loss.max(),1e-9) or (Dg.mean_abs_z.max()-Dg.mean_abs_z.min())>0.1
    print("VERDICT:", "γ moved the representation -> flat retention is a real refutation" if moved
          else "γ barely moved -> sweep underpowered; WIDEN γ (DECISIONS §D.5)")
else: print("[pending] need results/funnybirds-mcbm-g*/<seed>/predictions/epoch_100.pth")

📊 **γ genuinely bit** — the minimality term drops (rep_loss 458→~0.15, mean|z|→±3). So the flat backwash above is a real refutation, not an inert sweep (see DECISIONS §D.5).

## 5 · IB compression vs grounding — *your §20, deletion-adapted*
*(ports `fb_mcbm_renderer_swap.ipynb §20` "IB compression vs visual swap response")*.
The original plotted z-scale (compression) vs the part-**swap** response; we don't have
the live renderer in curated yet, so we use the **deletion** signal we do have: does IB
compressing `z` (z-std ↓, `mean|z|` up toward ±3) coincide with the tail becoming
**grounded** (retained_frac ↓)? If z compresses but retention stays flat, **compression
is not selective for grounding** — the §20 conclusion, on curated models.

In [ ]:
import torch
def zscale(cfg):
    vals=[]
    for seed in range(1,6):
        p=REPO/"external"/"minimal_cbm"/"results"/cfg/str(seed)/"predictions"/"epoch_100.pth"
        if not p.exists(): continue
        d=torch.load(p,map_location="cpu",weights_only=False); z,cc=d["z"].float(),d["c"].float()
        if not np.isfinite(z).all(): continue
        vals.append((float(z.std()), float(z.abs().mean())))
    return np.array(vals).mean(0) if vals else None
def tail_ret(tag):
    rf=[]
    for f in glob.glob(str(CURATED/"grounding"/f"funnybirds-mcbm-{tag}-s*.parquet")):
        d=pd.read_parquet(f); d=d[d.part=="tail"]
        if "changed_frac" in d.columns: d=d[d["changed_frac"]>EPS]
        if len(d) and d.p_intact.mean()>1e-6: rf.append(d.p_removed.mean()/d.p_intact.mean())
    return np.mean(rf) if rf else np.nan
rows=[]
for g,tag in [(0,"g0"),(0.1,"g0p1"),(0.3,"g0p3"),(1,"g1"),(3,"g3"),(5,"g5")]:
    zs=zscale(f"funnybirds-mcbm-{tag}"); tr=tail_ret(tag)
    if zs is not None and np.isfinite(tr): rows.append((g,zs[0],zs[1],tr))
if rows:
    D=pd.DataFrame(rows,columns=["gamma","z_std","mean_abs_z","tail_retained"]); display(D.round(3))
    floor=(D.gamma[D.gamma>0].min() or 0.05)/3
    fig,ax=plt.subplots(1,2,figsize=(12,4))
    ax[0].plot(D.gamma.replace(0,floor),D.z_std,"o-",color="steelblue",label="z std (spread)")
    a=ax[0].twinx(); a.plot(D.gamma.replace(0,floor),D.mean_abs_z,"s--",color="crimson",label="mean|z| (toward +-3)")
    ax[0].set_xscale("log"); ax[0].set_xlabel("gamma"); ax[0].set_ylabel("z std"); a.set_ylabel("mean|z|")
    ax[0].set_title("IB compresses z as gamma increases"); ax[0].legend(loc="upper left",fontsize=8); a.legend(loc="upper right",fontsize=8)
    sc=ax[1].scatter(D.z_std,D.tail_retained,c=D.gamma,cmap="viridis",s=90)
    for _,r in D.iterrows(): ax[1].annotate(f"g={r.gamma:g}",(r.z_std,r.tail_retained),fontsize=8)
    ax[1].set_xlabel("z std  (more compression <-)"); ax[1].set_ylabel("tail retained_frac")
    ax[1].set_title("Compression does NOT buy grounding (retention ~flat)"); plt.colorbar(sc,ax=ax[1],label="gamma")
    plt.tight_layout()
    print("z compresses with gamma but tail retained_frac stays ~flat -> IB compression is not selective for part grounding (section 20).")
else:
    print("[pending] need mcbm predictions (epoch_100.pth) + grounding parquets")

📊 z compresses as γ rises, but **tail retention stays flat** → IB compression is not selective for part grounding (your §20).

## 6 · Renderer-swap z-ordering vs γ — does minimality fix grounding?  *(ref §13–19)*
Same swap as CBM (notebook 02 §6), swept over the minimality strength γ. The question:
as the information bottleneck tightens, does the **tail** start correctly detecting the
swapped-in part (grounding restored), or does it stay backwashed? Reads the MCBM swap
CSVs `swap/funnybirds-mcbm-g*-s1.csv` (produced by
`CONFIG_PREFIX=funnybirds-mcbm sbatch train/renderer_swap.slurm`) + the CBM CSV as reference.

In [ ]:
ORDER=["tail","wing","beak","foot","eye"]
def load_swaps():
    rows=[]
    for fp in sorted(glob.glob(str(CURATED/"swap"/"funnybirds-mcbm-g*-s1.csv"))):
        m=re.search(r"-g([0-9p]+)-s",Path(fp).name)
        if not m: continue
        d=pd.read_csv(fp); d["gamma"]=float(m.group(1).replace("p",".")); rows.append(d)
    SW=pd.concat(rows,ignore_index=True) if rows else None
    cb=CURATED/"swap"/"funnybirds-cbm-s1.csv"
    return SW, (pd.read_csv(cb) if cb.exists() else None)
SW,CB=load_swaps()
if SW is None:
    print("[pending] no MCBM swap CSVs -> CONFIG_PREFIX=funnybirds-mcbm GAMMAS=\"0 0.1 0.3 1 3 5\" sbatch train/renderer_swap.slurm")
else:
    H=SW.groupby(["gamma","part"]).ordering_correct.mean().unstack().reindex(columns=ORDER)
    display(H.round(3))
    fig,ax=plt.subplots(figsize=(6.2,3.8)); im=ax.imshow(H.values,cmap="RdYlGn",vmin=0,vmax=1,aspect="auto")
    ax.set_xticks(range(len(ORDER))); ax.set_xticklabels(ORDER)
    ax.set_yticks(range(len(H.index))); ax.set_yticklabels([f"γ={g:g}" for g in H.index])
    for i in range(H.shape[0]):
        for j in range(H.shape[1]):
            v=H.values[i,j]
            if np.isfinite(v): ax.text(j,i,f"{v:.2f}",ha="center",va="center",fontsize=8)
    ax.set_title("z-ordering_correct — part × γ (green=grounded, red=backwash)"); fig.colorbar(im,fraction=0.046)

📊 **The headline (ref §19).** If the **tail column stays red down every γ**, tightening minimality does **not** restore tail grounding — MCBM inherits CBM's backwash. Green columns (foot/wing) = grounded at all γ.

In [ ]:
if SW is not None:
    t=SW[SW.part=="tail"].groupby("gamma").ordering_correct.mean()
    floor=(t.index[t.index>0].min() or 0.05)/3 if (t.index>0).any() else 0.02
    fig,ax=plt.subplots(figsize=(6,3.6))
    ax.plot([g if g>0 else floor for g in t.index], t.values,"o-",color=MCBM_C,label="MCBM tail")
    if CB is not None:
        ax.axhline(CB[CB.part=="tail"].ordering_correct.mean(),ls="--",color=CBM_C,label="CBM tail (ref)")
    ax.axhline(0.5,ls=":",color="gray"); ax.set_xscale("log"); ax.set_ylim(0,1.02)
    ax.set_xlabel("γ (minimality)"); ax.set_ylabel("tail ordering_correct"); ax.legend()
    ax.set_title("Does minimality fix the tail swap?")
else: print("[pending] mcbm swap CSVs")

📊 **Tail grounding vs γ.** Flat/low across γ (near the CBM dashed line, below the 0.5 coin-flip) = minimality does not fix it — the core MCBM refutation, now on the *causal* swap metric (not just deletion).

In [ ]:
if SW is not None and "pixel_count_cf" in SW.columns:
    allg=SW[SW.part=="tail"].groupby("gamma").ordering_correct.mean()
    visg=SW[(SW.part=="tail")&(SW.pixel_count_cf>=50)].groupby("gamma").ordering_correct.mean()
    D=pd.DataFrame({"all":allg,"visible_only":visg}); display(D.round(3))
    fig,ax=plt.subplots(figsize=(6,3.4)); x=np.arange(len(D))
    ax.plot(x,D["all"],"o-",label="all swaps",color="#9ec9e2")
    ax.plot(x,D["visible_only"],"s-",label="visible-only",color=MCBM_C)
    ax.set_xticks(x); ax.set_xticklabels([f"γ={g:g}" for g in D.index]); ax.set_ylim(0,1.02)
    ax.set_ylabel("tail ordering_correct"); ax.legend(); ax.set_title("Occlusion control for tail, per γ")
else: print("[pending] mcbm swap CSVs with pixel_count_cf")

📊 **Occlusion control across γ.** visible-only stays well below grounded at every γ → tail backwash survives both the visibility filter *and* minimality.

## Takeaway
If retention is flat/rising in γ while §4 shows γ tightened the rep and §3 shows species
stays recoverable, minimality shaped *content* but left the *class channel* intact.
**Lock only with ≥3 seeds + a bounded-CI (equivalence) statement — see `DECISIONS §D.5`.**

## How `retained_frac` is read as a backwash measurement
No axis is labelled "backwash"; the computed number is
**`retained_frac = P(concept | part removed) / P(concept | intact)`**, on **visible-only**
removals (the part actually left the render). Grounded → collapses to ~0; backwashed →
stays ~1. `retained_frac` is the metric; "concept–class backwash" is the interpretation.